First create zap bin profiles

In [ ]:
import glob
import psrchive
import numpy as np
import matplotlib
matplotlib.use("Agg")
# source /home/sangita/.bashrc_parkes
for pfd_file in glob.glob("*.pfd"):

    arch = psrchive.Archive_load(pfd_file)
    data = arch.get_data()
    # data shape: (nsubint, npol, nchan, nbin)

    nchan = arch.get_nchan()

    zapped_channels = []

    for ch in range(nchan):
        # channel is zapped if all bins are zero
        if np.all(data[0, 0, ch, :] == 0):
            zapped_channels.append(ch)

    # save zap list in ONE ROW (space-separated)
    zapfile = pfd_file.replace(".pfd", "_zaph.txt")
    with open(zapfile, "w") as f:
        f.write(" ".join(map(str, zapped_channels)))

    # print zap list
    print(f"\n{pfd_file}")
    print("Zapped channels:")
    print(zapped_channels)


This is for paper: Related to spectra study:

Same code, with adding mouse click for detecting on and off pusled region

In [ ]:
import glob
import psrchive
import numpy as np
import pandas as pd
import os
import json
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt

# -------------------------------
# Helper Functions
# -------------------------------

def read_zap_list(zapfile):
    if not os.path.exists(zapfile):
        return []
    with open(zapfile, "r") as f:
        content = f.read().strip()
        return list(map(int, content.split())) if content else []


def get_clean_channels(nchan, zap_list):
    return [ch for ch in range(nchan) if ch not in zap_list]


def split_into_subbands(channels, nsub=10):
    return np.array_split(channels, nsub)


def load_saved_windows(json_file):
    if os.path.exists(json_file):
        with open(json_file, "r") as f:
            return json.load(f)
    return {}


def save_windows(json_file, data):
    with open(json_file, "w") as f:
        json.dump(data, f, indent=2)


# -------------------------------
# Overlay plot
# -------------------------------

def plot_overlay(subbands, data):
    plt.figure(figsize=(8, 4))

    for i, subband in enumerate(subbands):
        profile = np.mean(data[subband, :], axis=0)
        profile = profile / np.max(profile)
        plt.plot(profile, label=f"S{i}", alpha=0.6)

    plt.title("Overlay of all subbands (normalized)")
    plt.xlabel("Bin")
    plt.ylabel("Normalized Intensity")
    plt.legend(ncol=2, fontsize=8)
    plt.grid(alpha=0.3)
    plt.show()


# -------------------------------
# Click function (ON/OFF)
# -------------------------------

def get_onpulse_by_click(profile, subband_id, integrated_profile=None, label="ON"):
    nbins = len(profile)

    while True:
        x = np.arange(nbins)

        peak_bin = np.argmax(profile)
        shift = nbins // 2 - peak_bin

        profile_shifted = np.roll(profile, shift)

        if integrated_profile is not None:
            integrated_shifted = np.roll(integrated_profile, shift)

            profile_shifted = profile_shifted / np.max(profile_shifted)
            integrated_shifted = integrated_shifted / np.max(integrated_shifted)

        fig, ax = plt.subplots(figsize=(10, 4))

        ax.plot(x, profile_shifted, label="Subband", color="C0")

        if integrated_profile is not None:
            ax.plot(x, integrated_shifted, '--', label="Integrated", color="C1", alpha=0.8)

        ax.set_title(f"Subband {subband_id}: Select {label}-pulse (shifted)")
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend()

        print(f"\nSubband {subband_id}: Select {label}-pulse region")

        pts = plt.ginput(2, timeout=-1)

        if len(pts) < 2:
            plt.close()
            raise RuntimeError("Need two clicks")

        start_s = int(round(pts[0][0]))
        end_s   = int(round(pts[1][0]))
        start_s, end_s = sorted([start_s, end_s])

        ax.plot(start_s, profile_shifted[start_s], 'ro')
        ax.plot(end_s, profile_shifted[end_s], 'ro')
        ax.axvspan(start_s, end_s, alpha=0.3)

        plt.draw()

        print(f"Selected (shifted): {start_s} → {end_s}")
        user = input("Press ENTER to confirm OR type anything to reselect: ")

        plt.close(fig)

        if user.strip() == "":
            break
        else:
            print("Redo selection...\n")

    start = int((start_s - shift) % nbins)
    end   = int((end_s   - shift) % nbins)

    print(f"Converted (original bins): {start} → {end}")

    return start, end


# -------------------------------
# Main Loop
# -------------------------------

for pfd_file in glob.glob("*.pfd"):

    print(f"\nProcessing: {pfd_file}")

    arch = psrchive.Archive_load(pfd_file)
    arch.dedisperse()
    arch.remove_baseline()

    data = arch.get_data().sum(axis=0)[0]
    nchan, nbins = data.shape
    freqs = arch.get_frequencies()

    zapfile = pfd_file.replace(".pfd", "_zaph.txt")
    zap_list = read_zap_list(zapfile)

    clean_channels = get_clean_channels(nchan, zap_list)
    clean_channels = sorted(clean_channels, key=lambda ch: freqs[ch], reverse=True)

    if len(clean_channels) < 10:
        continue

    subbands = split_into_subbands(clean_channels, 10)

    integrated_profile = np.mean(data[clean_channels, :], axis=0)

    print("\nShowing overlay of all subbands...")
    plot_overlay(subbands, data)

    json_file = pfd_file.replace(".pfd", "_onpulse_windows.json")
    saved_windows = load_saved_windows(json_file)

    use_saved = input("Use saved on-pulse windows if available? (y/n): ").lower()

    results = []
    windows_to_save = {}

    mode = input("Use common ON-pulse window? (y/n): ").lower()
    global_onpulse = None

    if mode == 'y':
        profile = np.mean(data[subbands[0], :], axis=0)
        global_onpulse = get_onpulse_by_click(profile, 0, integrated_profile, "ON")

    fig, axes = plt.subplots(10, 1, figsize=(6, 12), sharex=True)

    for i, subband in enumerate(subbands):

        profile = np.mean(data[subband, :], axis=0)

        # -------------------------------
        # ON pulse
        # -------------------------------
        if global_onpulse:
            start, end = global_onpulse
        else:
            start, end = get_onpulse_by_click(profile, i, integrated_profile, "ON")

        # -------------------------------
        # OFF pulse
        # -------------------------------
        off_start, off_end = get_onpulse_by_click(profile, i, integrated_profile, "OFF")

        # -------------------------------
        # Masks
        # -------------------------------
        on_mask = np.zeros(nbins, dtype=bool)
        on_mask[np.arange(start, start + (end - start) % nbins) % nbins] = True

        off_mask = np.zeros(nbins, dtype=bool)
        off_mask[np.arange(off_start, off_start + (off_end - off_start) % nbins) % nbins] = True

        # -------------------------------
        # Compute quantities
        # -------------------------------
        on_data = profile[on_mask]
        off_data = profile[off_mask]

        flux = np.mean(on_data)
        rms = np.std(off_data)
        peak = np.max(on_data)

        snr = peak / rms if rms > 0 else np.nan

        freq = np.mean(freqs[subband])

        results.append({
            "subband": int(i),
            "central_freq_MHz": float(freq),
            "on_start_bin": int(start),
            "on_end_bin": int(end),
            "off_start_bin": int(off_start),
            "off_end_bin": int(off_end),
            "mean_flux": float(flux),
            "rms_off": float(rms),
            "peak_on": float(peak),
            "snr": float(snr)
        })

        # -------------------------------
        # Plot
        # -------------------------------
        ax = axes[i]
        x = np.linspace(0, 1, nbins)

        ax.plot(x, profile)

        if start < end:
            ax.axvspan(x[start], x[end], alpha=0.3, color='green')
        else:
            ax.axvspan(x[start], 1, alpha=0.3, color='green')
            ax.axvspan(0, x[end], alpha=0.3, color='green')

        if off_start < off_end:
            ax.axvspan(x[off_start], x[off_end], alpha=0.2, color='red')
        else:
            ax.axvspan(x[off_start], 1, alpha=0.2, color='red')
            ax.axvspan(0, x[off_end], alpha=0.2, color='red')

        ax.set_ylabel(f"{freq:.1f} MHz", fontsize=8)

        ax.text(
            0.98, 0.85,
            f"SNR={snr:.2f}",
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=8
        )

    axes[-1].set_xlabel("Pulse Phase")
    plt.tight_layout()

    plt.savefig(pfd_file.replace(".pfd", "_final.pdf"))
    plt.close()

    pd.DataFrame(results).to_csv(
        pfd_file.replace(".pfd", "_spectrum.csv"),
        index=False
    )

    print("Saved spectrum with SNR.")

testing above code

In [ ]:
import glob
import psrchive
import numpy as np
import pandas as pd
import os
import json
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt

# -------------------------------
# SAFE JSON CONVERSION
# -------------------------------

def convert_to_builtin(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, list):
        return [convert_to_builtin(i) for i in obj]
    elif isinstance(obj, tuple):
        return [convert_to_builtin(i) for i in obj]
    elif isinstance(obj, dict):
        return {k: convert_to_builtin(v) for k, v in obj.items()}
    else:
        return obj


def save_json(f, d):
    with open(f, "w") as fp:
        json.dump(convert_to_builtin(d), fp, indent=2)


def load_json(f):
    return json.load(open(f)) if os.path.exists(f) else {}


# -------------------------------
# HELPER
# -------------------------------

def read_zap_list(zapfile):
    if not os.path.exists(zapfile):
        return []
    return list(map(int, open(zapfile).read().split()))


def get_clean_channels(nchan, zap):
    return [i for i in range(nchan) if i not in zap]


def split_into_subbands(channels, nsub=10):
    return np.array_split(channels, nsub)


# -------------------------------
# PLOT
# -------------------------------

def plot_overlay(subbands, data):
    plt.figure(figsize=(8,4))
    for sb in subbands:
        p = np.mean(data[sb,:], axis=0)
        plt.plot(p/np.max(p), alpha=0.6)
    plt.show()


# -------------------------------
# INTERACTIVE SELECTION
# -------------------------------

def select_region(profile, integrated, label):
    nbins = len(profile)

    while True:
        shift = nbins//2 - np.argmax(profile)

        p = np.roll(profile, shift)
        ip = np.roll(integrated, shift)

        p /= np.max(p)
        ip /= np.max(ip)

        fig, ax = plt.subplots()
        ax.plot(p, label="Subband")
        ax.plot(ip, '--', label="Integrated")
        ax.set_title(f"{label} selection")
        ax.legend()

        pts = plt.ginput(2, timeout=-1)
        s, e = sorted([int(pts[0][0]), int(pts[1][0])])

        ax.axvspan(s,e,alpha=0.3)
        plt.draw()

        ok = input("ENTER confirm / other redo: ")
        plt.close(fig)

        if ok.strip()=="":
            break

    return int((s-shift)%nbins), int((e-shift)%nbins)


def select_multiple_off(profile, integrated):
    regions = []
    while True:
        r = select_region(profile, integrated, "OFF")
        regions.append(r)
        if input("More OFF? (ENTER=yes, q=stop): ").lower()=="q":
            break
    return regions


# -------------------------------
# MAIN
# -------------------------------

for f in glob.glob("*.pfd"):

    print("\nProcessing:", f)

    arch = psrchive.Archive_load(f)
    arch.dedisperse()
    arch.remove_baseline()

    data = arch.get_data().sum(axis=0)[0]
    nchan, nbins = data.shape
    freqs = arch.get_frequencies()

    zap = read_zap_list(f.replace(".pfd","_zaph.txt"))
    clean = sorted(get_clean_channels(nchan, zap), key=lambda x: freqs[x], reverse=True)

    if len(clean)<10:
        continue

    subbands = split_into_subbands(clean,10)
    integrated = np.mean(data[clean,:], axis=0)

    plot_overlay(subbands, data)

    json_file = f.replace(".pfd","_windows.json")
    saved = load_json(json_file)

    use_saved = input("Use saved windows? (y/n): ").lower()

    # -------------------------------
    # USE SAVED
    # -------------------------------
    if use_saved=='y' and "on" in saved and "off" in saved:
        print("Using saved ON/OFF windows.")
        global_on = tuple(saved["on"])
        global_off = [tuple(x) for x in saved["off"]]

        use_global_on = 'y'
        use_global_off = 'y'

    # -------------------------------
    # NEW SELECTION
    # -------------------------------
    else:
        use_global_on = input("Same ON for all subbands? (y/n): ").lower()
        use_global_off = input("Same OFF for all subbands? (y/n): ").lower()

        global_on = None
        global_off = None

        if use_global_on=='y':
            profile = np.mean(data[subbands[0],:], axis=0)
            global_on = select_region(profile, integrated, "ON")

        if use_global_off=='y':
            profile = np.mean(data[subbands[0],:], axis=0)
            global_off = select_multiple_off(profile, integrated)

    results=[]

    # -------------------------------
    # LOOP
    # -------------------------------

    for i,sb in enumerate(subbands):

        profile = np.mean(data[sb,:], axis=0)

        # ON
        if use_global_on=='y':
            on_start, on_end = global_on
        else:
            on_start, on_end = select_region(profile, integrated, "ON")

        # OFF
        if use_global_off=='y':
            off_regions = global_off
        else:
            off_regions = select_multiple_off(profile, integrated)

        # MASK
        def mask_range(s,e):
            m = np.zeros(nbins,bool)
            m[np.arange(s, s+(e-s)%nbins)%nbins]=True
            return m

        on_data = profile[mask_range(on_start,on_end)]

        rms_list=[]
        for s,e in off_regions:
            d = profile[mask_range(s,e)]
            if len(d)>0:
                rms_list.append(np.std(d))

        rms = np.sqrt(np.sum(np.array(rms_list)**2)) if rms_list else np.nan
        peak = np.max(on_data)
        flux = np.mean(on_data)
        snr = peak/rms if rms>0 else np.nan

        results.append({
            "subband":i,
            "central_freq_MHz":float(np.mean(freqs[sb])),
            "on_start":on_start,
            "on_end":on_end,
            "off_regions":str(off_regions),
            "flux":float(flux),
            "rms":float(rms),
            "snr":float(snr)
        })

    df_new = pd.DataFrame(results)

    out_csv = f.replace(".pfd","_spectrum.csv")

    # -------------------------------
    # SAFE CSV SAVE (NO DATA LOSS)
    # -------------------------------
    if os.path.exists(out_csv):
        df_old = pd.read_csv(out_csv)

        # Merge without losing old columns
        df_final = pd.merge(df_old, df_new, on="subband", how="outer", suffixes=("_old",""))
    else:
        df_final = df_new

    df_final.to_csv(out_csv, index=False)

    # -------------------------------
    # SAVE WINDOWS
    # -------------------------------
    save_json(json_file, {"on":global_on, "off":global_off})

    print("Saved safely without overwriting.")

Spectra plot

After choosing the points, it should save the final plot and also update in the csv file by adding another column of 0 (not acocunt for spectra calculation) or 1 (account for spctra calculation)

Final first part code

In the interactive plot, add an otion for two power law or one power law fit, then accordingly interctively choose points for one or two power law fi

🖱️ Interactive behavior
▶ Single power-law:
Click points → include/exclude (as before)
▶ Broken power-law:
Click points → include/exclude
Right-click → select break region (two clicks)


Final working partial code for spectra plot Code

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
import os

# -------------------------------
# Load data (PER FILE)
# -------------------------------
files = glob.glob("J*_spectrum.csv")

for f in files:

    print(f"\nProcessing: {f}")

    df = pd.read_csv(f)

    grouped = df.groupby("central_freq_MHz")

    freq = grouped["central_freq_MHz"].mean().values
    flux = grouped["mean_flux"].mean().values
    flux_err = grouped["mean_flux"].std().values

    # Fix NaN errors
    flux_err = np.where(np.isnan(flux_err), 0.1 * flux, flux_err)

    # Keep valid
    mask = flux > 0
    freq = freq[mask]
    flux = flux[mask]
    flux_err = flux_err[mask]

    # -------------------------------
    # STEP 1: PREVIEW PLOT
    # -------------------------------
    plt.figure(figsize=(8,5))
    plt.errorbar(freq, flux, yerr=flux_err, fmt='o')

    plt.xscale("log")
    plt.yscale("log")

    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Flux")
    plt.title(f"Preview Spectrum: {os.path.basename(f)}")

    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # -------------------------------
    # STEP 2: MODEL SELECTION
    # -------------------------------
    mode = input("\nChoose model:\n1 = Single power-law\n2 = Broken power-law\nEnter choice: ").strip()

    # -------------------------------
    # Interactive setup
    # -------------------------------
    selected = np.ones(len(freq), dtype=bool)

    fig, ax = plt.subplots(figsize=(7,6))

    sc = ax.scatter(freq, flux, picker=True)
    ax.errorbar(freq, flux, yerr=flux_err, fmt='none')

    fit_lines = []

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Flux")
    ax.grid(True, which='both', ls='--', alpha=0.5)

    text = ax.text(0.05, 0.95, "", transform=ax.transAxes, va='top')

    # -------------------------------
    # Fit functions
    # -------------------------------
    def single_fit():
        sel = selected
        if np.sum(sel) < 2:
            return None

        f_sel = freq[sel]
        S = flux[sel]
        err = flux_err[sel]

        logf = np.log10(f_sel)
        logs = np.log10(S)

        w = 1 / (err / S)
        c = np.polyfit(logf, logs, 1, w=w)

        return c


    def broken_fit():
        sel = selected
        if np.sum(sel) < 4:
            return None

        f_sel = freq[sel]
        S = flux[sel]

        f_break = np.median(f_sel)

        left = f_sel <= f_break
        right = f_sel > f_break

        c1 = np.polyfit(np.log10(f_sel[left]), np.log10(S[left]), 1)
        c2 = np.polyfit(np.log10(f_sel[right]), np.log10(S[right]), 1)

        return c1, c2, f_break


    # -------------------------------
    # Update plot (NO global/nonlocal)
    # -------------------------------
    def update_fit():

        # remove previous lines safely
        while fit_lines:
            fit_lines.pop().remove()

        if mode == "1":
            c = single_fit()
            if c is None:
                return

            alpha = c[0]
            f_fit = np.linspace(min(freq), max(freq), 100)
            S_fit = 10**c[1] * f_fit**alpha

            line, = ax.plot(f_fit, S_fit)
            fit_lines.append(line)

            text.set_text(f"Single PL: α = {alpha:.2f}")

        else:
            result = broken_fit()
            if result is None:
                return

            c1, c2, f_break = result

            f1 = np.linspace(min(freq), f_break, 50)
            f2 = np.linspace(f_break, max(freq), 50)

            S1 = 10**c1[1] * f1**c1[0]
            S2 = 10**c2[1] * f2**c2[0]

            l1, = ax.plot(f1, S1, 'g')
            l2, = ax.plot(f2, S2, 'g')

            fit_lines.extend([l1, l2])

            text.set_text(f"Broken PL:\nα1={c1[0]:.2f}, α2={c2[0]:.2f}")

        fig.canvas.draw_idle()


    # -------------------------------
    # Click handler
    # -------------------------------
    def on_pick(event):
        ind = event.ind[0]
        selected[ind] = ~selected[ind]

        colors = ['red' if not s else 'blue' for s in selected]
        sc.set_color(colors)

        update_fit()


    fig.canvas.mpl_connect('pick_event', on_pick)

    update_fit()

    plt.title(f"Click points: {os.path.basename(f)}")
    plt.show()

    # -------------------------------
    # Save results (PER FILE)
    # -------------------------------
    out_csv = f.replace("_spectrum.csv", "_model_selection.csv")

    df_out = pd.DataFrame({
        "central_freq_MHz": freq,
        "mean_flux": flux,
        "flux_err": flux_err,
        "use_for_fit": selected.astype(int)
    })

    # -------------------------------
    # Extract alpha values
    # -------------------------------
    alpha_list = []

    if mode == "1":
        c = single_fit()
        if c is not None:
            alpha_list.append(("single", float(c[0])))

    else:
        result = broken_fit()
        if result is not None:
            c1, c2, _ = result
            alpha_list.append(("broken", float(c1[0])))
            alpha_list.append(("broken", float(c2[0])))

    # -------------------------------
    # Save alpha values
    # -------------------------------
    alpha_file = f.replace("_spectrum.csv", "_alpha.csv")

    df_alpha = pd.DataFrame(alpha_list, columns=["model", "alpha"])

    df_alpha.to_csv(alpha_file, index=False)

    print(f"Saved: {alpha_file}")

    df_out.to_csv(out_csv, index=False)

    # Save plot
    fig.savefig(f.replace("_spectrum.csv", "_fit.png"))

    print(f"Saved: {out_csv}")